# Thêm Thư Viện

In [23]:
import pyodbc
import pandas as pd
import numpy as np

# Tạo kết nối

In [24]:
conn_dwh_library = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=192.168.150.6;'  # Địa chỉ IP của SQL Server
    'DATABASE=dwh_library;'         # Tên cơ sở dữ liệu
    'UID=itc;'                # Tên đăng nhập
    'PWD=spkt@2025;'
)
conn_Library_DWH = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=192.168.150.6;' # Địa chỉ IP của SQL Server
    'DATABASE=Library_DWH;' # Tên cơ sở dữ liệu
    'UID=itc;'              # Tên đăng nhập
    'PWD=spkt@2025;'
)

# Đọc data

## Đọc data từ SQL Server

In [ ]:
query_DIM_phieu_muon = """
SELECT ID_phieu_muon,
      ID_tai_lieu,
      ID_xep_gia,
      ID_ban_doc,
      Ngay_muon,
      Ngay_tra,
      So_luot_gia_han,
      So_ngay_qua_han,
      Tien_phat,
      Ghi_chu
FROM DIM_Phieu_muon_sach
"""
df_phieumuon = pd.read_sql(query_DIM_phieu_muon, conn_dwh_library)
print(df_phieumuon)

     ID_phieu_muon  ID_tai_lieu  ID_xep_gia ID_ban_doc  Ngay_muon  Ngay_tra  \
0                1        11089      241160          0   20010219  20080219   
1                2        11096      315718          0   20010219  20080219   
2                3        22194      485966          0   20010219  20080219   
3                4        14668      314619          0   20010219  20080219   
4                5        12495      277035          0   20010219  20080219   
..             ...          ...         ...        ...        ...       ...   
177            178         3950      209935          0   20020930  20021002   
178            179         2656      206099          0   20020930  20021002   
179            180         2738           0          0   20020930  20021002   
180            181         3608      209153          0   20020930  20020930   
181            182         4024           0          0   20020930  20020930   

    So_luot_gia_han  So_ngay_qua_han  Tien_phat Ghi

C:\Users\admin\AppData\Local\Temp\ipykernel_22664\582981565.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_phieumuon = pd.read_sql(query_DIM_phieu_muon, conn_dwh_library)


### [Nếu cần] Clear bảng

In [26]:
cursor = conn_Library_DWH.cursor()
truncate_query = "DELETE FROM oltp.Phieu_muon_sach"
cursor.execute(truncate_query)
conn_Library_DWH.commit()
cursor.close()

### Load data vào bảng Dim

In [21]:
# Tạo cursor để thao tác với cơ sở dữ liệu
cursor_dwh = conn_Library_DWH.cursor()

# Chuẩn bị câu lệnh chèn dữ liệu
insert_query = """
                INSERT INTO oltp.Phieu_muon_sach (
                    ID_phieu_muon, 
                    ID_tai_lieu, ID_xep_gia,
                    ID_ban_doc, 
                    Ngay_muon, Ngay_tra, 
                    So_luot_gia_han, So_ngay_qua_han, 
                    Tien_phat, Ghi_chu
                ) 
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
               """
# Chuyển đổi dữ liệu từ DataFrame thành danh sách các tuple để chèn
data_to_insert = [
    (
        row['ID_phieu_muon'], 
        row['ID_tai_lieu'], row['ID_xep_gia'], 
        row['ID_ban_doc'],
        row['Ngay_muon'], row['Ngay_tra'], 
        row['So_luot_gia_han'], row['So_ngay_qua_han'], 
        row['Tien_phat'], row['Ghi_chu']
    )
    for index, row in df_phieumuon.iterrows()
]
# Sử dụng executemany để chèn dữ liệu cùng lúc
cursor_dwh.executemany(insert_query, data_to_insert)
# Commit thay đổi
conn_Library_DWH.commit()
# Đóng cursor và kết nối
cursor_dwh.close()
conn_Library_DWH.close()
